In [ ]:
!pip install -q langchain langchain-openai

In [ ]:
from google.colab import userdata
from langchain.agents import create_agent
from langchain.agents.middleware import ContextEditingMiddleware, ModelRequest, ModelResponse, wrap_model_call
from langchain.agents.middleware.context_editing import ClearToolUsesEdit
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain_core.messages import BaseMessage
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import SecretStr
from typing import Callable, List

api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

In [ ]:
@tool
def fetch_logs(service: str) -> str:
    """Fetch raw application logs for a service over the last N minutes."""
    line = "[2026-04-20T10:15:{s:02d}Z] {service} WARN db pool exhausted retries=3 latency_ms=842 trace_id=abc123 user_id=U-{s:04d}"
    return "\n".join(line.format(s=i % 60, service=service) for i in range(10))

In [ ]:
@wrap_model_call
def log_model_input(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]):
    print(f"\n--- model sees {len(request.messages)} messages ---")
    for m in request.messages:
        preview = str(getattr(m, "content", ""))[:60].replace("\n", " ")
        extra = f"tool_calls={len(m.tool_calls)}" if hasattr(m, "tool_calls") else ""
        output = [f"{m.type:12s}", preview, extra]

        print(" | ".join(x for x in output if x != ""))

    return handler(request)

In [ ]:
model = ChatOpenAI(model="gpt-5-nano", api_key=api_key, reasoning_effort="low")

agent = create_agent(
    model=model,
    tools=[fetch_logs],
    system_prompt=f"You are an on-call SRE assistant. For any incident question, first call `{fetch_logs.name}` for the relevant service, then summarize root-cause signals in plain language. Be concise.",
    checkpointer=InMemorySaver(),
    middleware=[
        ContextEditingMiddleware(
            edits=[
                ClearToolUsesEdit(trigger=200, keep=1, clear_tool_inputs=True)
            ]
        ),
        log_model_input
    ]
)

interact = agent | RunnableLambda(lambda res: print_conversation(res["messages"]))

Review the output of each following cell through to the end of the notebook.
At some point, earlier tool calls will be cleared to free up context.

In [ ]:
interact.invoke(
    input={
        "messages": [HumanMessage("Fetch logs for the 'application_api' service.")]
    },
    config={
        "configurable": {
            "thread_id": "thread_1"
        }
    }
)

In [ ]:
interact.invoke(
    input={
        "messages": [HumanMessage("Let's explore the 'administration_api' service now.")]
    },
    config={
        "configurable": {
            "thread_id": "thread_1"
        }
    }
)

In [ ]:
interact.invoke(
    input={
        "messages": [HumanMessage("I need the logs for two more services - 'auth_api' and 'payment_gateway'.")]
    },
    config={
        "configurable": {
            "thread_id": "thread_1"
        }
    }
)

In [ ]:
interact.invoke(
    input={
        "messages": [HumanMessage("And one final request - the logs from the 'analytics_api'.")]
    },
    config={
        "configurable": {
            "thread_id": "thread_1"
        }
    }
)